In [ ]:
#task1

import re
pattern1 = re.compile(r'abc')
result1 = pattern1.match('abcdef')
if result1:
  print("Match found:", result1.group())
pattern2 = re.compile(r'.')
result2 = pattern2.search('Hello')
if result2:
  print("Character found:", result2.group())
pattern3 = re.compile(r'[aeiou]')
result3 = pattern3.search('Hello')
if result3:
  print("Vowel found:", result3.group())
pattern4 = re.compile(r'\d{3}-\d{2}-\d{4}')
result4 = pattern4.match('123-45-6789')
if result4:
  print("Social Security Number:", result4.group())
pattern5 = re.compile(r'\.')
result5 = pattern5.search('www.example.com')
if result5:
  print("Dot found:", result5.group())
pattern6 = re.compile(r'(\d{2})/(\d{2})/(\d{4})')
result6 = pattern6.match('01/09/2024')
if result6:
  print("Day:", result6.group(1))
  print("Month:", result6.group(2))
  print("Year:", result6.group(3))
pattern7 = re.compile(r'cat|dog')
result7 = pattern7.search('I have a cat and a dog')
if result7:
  print("Animal found:", result7.group())

Match found: abc
Character found: H
Vowel found: e
Social Security Number: 123-45-6789
Dot found: .
Day: 01
Month: 09
Year: 2024
Animal found: cat


In [ ]:
#Task 2
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
text = "Tokenization without transformers is straightforward with tools like NLTK."
tokens = word_tokenize(text)
print("Tokens:", tokens)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Tokenization without transformers is straightforward with tools like NLTK."
tokens_transformers = tokenizer(text, return_tensors="pt")
print("Transformers Tokens:", tokens_transformers)
tokens_transformers_list = tokenizer.convert_ids_to_tokens(tokens_transformers['input_ids'][0].numpy().tolist())
print("Transformers Tokens (List):", tokens_transformers_list)
decoded_text = tokenizer.decode(tokens_transformers['input_ids'][0], skip_special_tokens=True)
print("Decoded Text:", decoded_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokens: ['Tokenization', 'without', 'transformers', 'is', 'straightforward', 'with', 'tools', 'like', 'NLTK', '.']


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Transformers Tokens: {'input_ids': tensor([[  101, 19204,  3989,  2302, 19081,  2003, 19647,  2007,  5906,  2066,
         17953,  2102,  2243,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Transformers Tokens (List): ['[CLS]', 'token', '##ization', 'without', 'transformers', 'is', 'straightforward', 'with', 'tools', 'like', 'nl', '##t', '##k', '.', '[SEP]']
Decoded Text: tokenization without transformers is straightforward with tools like nltk.


In [ ]:
#task 3
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
import spacy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('punkt')
nltk.download('punkt_tab')
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

corpus = """One disadvantage of using 'Best Of' samping is that it may lead to limited exploration of the model's
knowledge and creativity. By focusing on the most probable next words, the model might generate responses that are
safe and conventional, potentially missing out on more diverse and innovative outputs. The lack of exploration could
result in repetitive or less imaginative responses, especially in situations where novel and unconventional ideas are
desired.To address this limitation, other sampling strategies like temperature-based sampling or top-p (nucleus) sampling
can be employed to introduce more randomness and encourage the model to explore a broader range of possibilities.
However, it's essential to carefully balance exploration and exploitation based on the specific requirements of the task or
application."""

tokens = word_tokenize(corpus)
lemmatized_tokens = [token.lemma_ for token in nlp(corpus)]

all_tokens = tokens + lemmatized_tokens

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_tokens)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for token in all_tokens:
    token_list = tokenizer.texts_to_sequences([token])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

input_sequences = []
for i in range(1, len(all_tokens)):
    token_list = tokenizer.texts_to_sequences(all_tokens[:i+1])
    token_list = [item for sublist in token_list for item in sublist]
    input_sequences.append(token_list)

max_sequence_length = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')

X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = np.array(y)


model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_length-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X, y, epochs=5, verbose=1)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 13s 566ms/step - accuracy: 0.0274 - loss: 4.6707
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 360ms/step - accuracy: 0.1158 - loss: 4.6457
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 583ms/step - accuracy: 0.0579 - loss: 4.5354
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 482ms/step - accuracy: 0.0527 - loss: 4.4018
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 434ms/step - accuracy: 0.0465 - loss: 4.3166
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 403ms/step - accuracy: 0.0945 - loss: 4.2769
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 341ms/step - accuracy: 0.1036 - loss: 4.0949
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 354ms/step - accuracy: 0.0884 - loss: 3.9395
Epoch 9/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 408ms/step - accuracy: 0.0912 - loss: 3.8879
Epoch 10/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 467ms/step - accuracy: 0.1332 - loss: 3.7176


In [ ]:
#task 4
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
nltk.download('punkt')
nltk.download('stopwords')
def tokenize_document(document):
    tokens = word_tokenize(document)
    return [word.lower() for word in tokens if word.isalpha()]
def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [word for word in tokens if word not in stop_words]
def find_morphology(tokens):
    fdist = FreqDist(tokens)
    return fdist.most_common(10)
document_text =""" Technology has changed the way we live, work, and communicate.
From smartphones to artificial intelligence,it continues to shape our daily lives.
In education, online learning platforms have made knowledge more accessible than ever.
Healthcare has also benefited, with advanced tools improving diagnosis and treatment.
Businesses use technology to streamline operations and reach global markets.
Social media connects people across continents, though it also raises concerns about privacy and mental health.
Despite its many advantages, technology can widen the gap between those with access and those without.
Environmental impacts, such as e-waste and energy use, must also be addressed.
As society grows more reliant on technology, ethical and responsible development becomes crucial.
Balancing progress with sustainability is one of the biggest challenges of our time.
"""
tokens = tokenize_document(document_text)
tokens_without_stopwords = remove_stopwords(tokens)
morphology = find_morphology(tokens_without_stopwords)
print("Morphology of the document:")
for word, frequency in morphology:
    print(f"{word}: {frequency}")

Morphology of the document:
technology: 4
also: 3
use: 2
changed: 1
way: 1
live: 1
work: 1
communicate: 1
smartphones: 1
artificial: 1


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#task 5

import string
import random
import nltk
from nltk.corpus import stopwords, reuters
from collections import Counter, defaultdict
from nltk import FreqDist, ngrams
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('reuters')
sents = reuters.sents()
stop_word = set(stopwords.words('english'))
extra_punct = set(['"', '-', '_'])
removal_list = set(stop_word) | set(string.punctuation) | extra_punct | set(['\t', 'rt'])
unigram = []
bigram = []
trigram = []
for sentence in sents:
    sentence = [word.lower() for word in sentence]
    sentence = [word for word in sentence if word != '.']
    unigram.extend(sentence)
    bigram.extend(list(ngrams(sentence, 2, pad_left=True, pad_right=True)))
    trigram.extend(list(ngrams(sentence, 3, pad_left=True, pad_right=True)))
def remove_stopwords(ngram_list):
    filtered = []
    for ngram in ngram_list:
        if any(word is None for word in ngram):
            filtered.append(ngram)
        else:
            if not any(word in removal_list for word in ngram):
                filtered.append(ngram)
    return filtered
unigram = [word for word in unigram if word not in removal_list]
bigram = remove_stopwords(bigram)
trigram = remove_stopwords(trigram)
freq_uni = FreqDist(unigram)
freq_bi = FreqDist(bigram)
freq_tri = FreqDist(trigram)
d = defaultdict(Counter)
for (a, b, c), freq in freq_tri.items():
    if a is not None and b is not None and c is not None:
        d[(a, b)][c] += freq
def pick_word(counter):
    "The sun was shining brightly over the hills.Birds chirped happily as the morning breeze blew gently."
    words, weights = zip(*counter.items())
    return random.choices(words, weights=weights)[0]
prefix = ("he", "is")
print(" ".join(prefix))
s = " ".join(prefix)
for i in range(19):
    counter = d[prefix]
    if not counter:
        prefix = random.choice(list(d.keys()))
        counter = d[prefix]
    suffix = pick_word(counter)
    s += ' ' + suffix
    prefix = (prefix[1], suffix)
print(s)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


he is
he is mths shr loss 1 10 billion crown contract morbelli mln revs 28 2 mln revs 348 8 mln dlrs


In [ ]:
#Task 6
from nltk.util import ngrams
from nltk.lm import Laplace
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
def ngram_smoothing(sentence, n):
  tokens = word_tokenize(sentence.lower())
  train_data, padded_sents = padded_everygram_pipeline(n, tokens)
  model = Laplace(n)
  model.fit(train_data, padded_sents)
  return model

sentence = input("Enter a sentence: ")
n = int(input("Enter the value of N for N-grams: "))
model = ngram_smoothing(sentence, n)
context = tuple(sentence.lower().split()[-n+1:])
next_words = model.generate(3, text_seed=context)
print("Next words:", ' '.join(next_words))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Enter a sentence: hi hello good morinng
Enter the value of N for N-grams: 4
Next words: l l o


In [ ]:
# Task 6
from nltk.util import ngrams
from nltk.lm import Laplace
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
import nltk

# Download required tokenizer models
nltk.download('punkt')
nltk.download('punkt_tab')

def ngram_smoothing(sentence, n):
    tokens = word_tokenize(sentence.lower())
    # Wrap tokens in a list to simulate a corpus with one sentence
    train_data, padded_sents = padded_everygram_pipeline(n, [tokens])
    model = Laplace(n)
    model.fit(train_data, padded_sents)
    return model

sentence = input("Enter a sentence: ")
n = int(input("Enter the value of N for N-grams: "))

model = ngram_smoothing(sentence, n)

# Extract context: last n-1 tokens from the sentence
context = tuple(sentence.lower().split()[-(n-1):])

# Generate next 3 words based on context
next_words = model.generate(3, text_seed=context)

print("Next words:", ' '.join(next_words))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Enter a sentence: hii
Enter the value of N for N-grams: 1
Next words: hii hii hii


In [ ]:
#Task 8
import nltk
from nltk.corpus import treebank
from nltk.tag import hmm
from nltk.classify import MaxentClassifier
!pip install -U nltk
nltk.download('maxent_treebank_pos_tagger')
from nltk.tag import PerceptronTagger, StanfordTagger
nltk.download('treebank')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('averaged_perceptron_tagger')
corpus = list(treebank.tagged_sents())
train_data = corpus[:int(0.8 * len(corpus))]
test_data = corpus[int(0.8 * len(corpus)):]
hmm_tagger = hmm.HiddenMarkovModelTrainer().train(train_data)
hmm_accuracy = hmm_tagger.evaluate(test_data)
print(f"HMM Tagger Accuracy: {hmm_accuracy:.4f}")

[nltk_data] Downloading package maxent_treebank_pos_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/maxent_treebank_pos_tagger.zip.
[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
/tmp/ipython-input-1367130732.py:17: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  hmm_accuracy = hmm_tagger.evaluate(test_data)
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].lo

HMM Tagger Accuracy: 0.3647
